In [15]:
import os
import requests
import time
import math
from dataclasses import dataclass
import pickle # For saving/loading meta later if needed, though not strictly for tokenizer now

import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
from tqdm import tqdm # For progress bars

# --- Hyperparameters ---
BATCH_SIZE = 64
BLOCK_SIZE = 256  # Context length
MAX_ITERS = 5000
EVAL_INTERVAL = 250
LEARNING_RATE = 3e-4 # Adjusted from 1e-3, often 3e-4 is a good starting point
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_EMBD = 384
N_HEAD = 6
N_LAYER = 6
DROPOUT = 0.2
# AdamW optimizer betas
BETA1 = 0.9
BETA2 = 0.95
# Early stopping
EARLY_STOPPING_PATIENCE = 5 # Number of evaluation intervals to wait
EVAL_ITERS_FOR_LOSS = 100 # Number of batches to average for loss estimation

# For reproducibility
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1337)

print(f"Using device: {DEVICE}")

Using device: cuda


In [16]:
# --- 1. Data Loading and Tokenization ---


input_file_path = 'fake_news_formatted.txt'
data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'

if not os.path.exists(input_file_path):
    print(f"Downloading {input_file_path}...")
    with open(input_file_path, 'w', encoding='utf-8') as f:
        f.write(requests.get(data_url).text)
    print("Download complete.")
else:
    print(f"{input_file_path} already exists.")

with open(input_file_path, 'r', encoding='utf-8') as f:
    data = f.read()
print(f"Length of dataset in characters: {len(data):,}")

# Get all unique characters
chars = sorted(list(set(data)))
vocab_size = len(chars)
print("All unique characters:", ''.join(chars))
print(f"Vocabulary size: {vocab_size:,}")

# Create mappings from characters to integers and vice-versa
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# Encoder: take a string, output a list of integers
def encode(s):
    return [stoi[c] for c in s]

# Decoder: take a list of integers, output a string
def decode(l):
    return ''.join([itos[i] for i in l])

# Create train and validation splits
n = len(data)
train_text = data[:int(n * 0.9)]
val_text = data[int(n * 0.9):]

# Encode the text and convert to PyTorch tensors
train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data = torch.tensor(encode(val_text), dtype=torch.long)

print(f"Train data has {len(train_data):,} tokens")
print(f"Validation data has {len(val_data):,} tokens")

fake_news_formatted.txt already exists.
Length of dataset in characters: 17,281,995
All unique characters: 
 !"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_`abcdefghijklmnopqrstuvwxyz{|}~£§«­°´¹»½¿ÁÅàáâãäåçèéíïñóôö÷øùúüāćėğōžΞВПСТабвгдежзийклмнопрстухцчшщыьэюя،؟ءأؤإئابةتثجحخدذرزسشصضطظعغفقكلمنهوىيὈ​‎‏–—‘’‚“”•…‼€■☑♔♥⚡⚾⛏❗❤➡ムﬁ️🇱🇲🇸🇹🇺🇽🌍🌎🌏🌙🌟🍑🎉🎙🎶🏻🏽🏾👇👊👏👨👽💔💪💯🖤🗽😂😈😍😏😔😩😬😭😯😱🙃🙏🚽
Vocabulary size: 275
Train data has 15,553,795 tokens
Validation data has 1,728,200 tokens


In [17]:
# --- Tokenizer Experimentation ---
sample_text = "Hello World This is a test."
print(f"Original text: '{sample_text}'")

encoded_sample = encode(sample_text)
print(f"Encoded text: {encoded_sample}")

decoded_sample = decode(encoded_sample)
print(f"Decoded text: '{decoded_sample}'")

assert sample_text == decoded_sample, "Encoding/Decoding mismatch!"
print("Tokenizer test passed!")

Original text: 'Hello World This is a test.'
Encoded text: [41, 68, 75, 75, 78, 1, 56, 78, 81, 75, 67, 1, 53, 71, 72, 82, 1, 72, 82, 1, 64, 1, 83, 68, 82, 83, 15]
Decoded text: 'Hello World This is a test.'
Tokenizer test passed!


In [18]:
# --- Data Loader ---
def get_batch(split):
    # Selects the appropriate dataset (train or val)
    data_source = train_data if split == 'train' else val_data
    # Generates random starting indices for batch_size sequences
    ix = torch.randint(len(data_source) - BLOCK_SIZE, (BATCH_SIZE,))
    # Extracts input sequences (x)
    x = torch.stack([data_source[i:i+BLOCK_SIZE] for i in ix])
    # Extracts target sequences (y), which are shifted by one character
    y = torch.stack([data_source[i+1:i+BLOCK_SIZE+1] for i in ix])
    # Move data to the specified device (CPU or GPU)
    x, y = x.to(DEVICE), y.to(DEVICE)
    return x, y

# Test get_batch
xb, yb = get_batch('train')
print("Input batch shape:", xb.shape)
print("Target batch shape:", yb.shape)
print("First sequence in input batch (first 30 tokens):", decode(xb[0][:30].tolist()))
print("First sequence in target batch (first 30 tokens):", decode(yb[0][:30].tolist()))

Input batch shape: torch.Size([64, 256])
Target batch shape: torch.Size([64, 256])
First sequence in input batch (first 30 tokens): territory, Delattre told the c
First sequence in target batch (first 30 tokens): erritory, Delattre told the co


In [19]:
# --- 2. Model Definition ---

@dataclass
class GPTConfig:
    block_size: int = BLOCK_SIZE
    vocab_size: int = vocab_size # Will be set by loaded data
    n_layer: int = N_LAYER
    n_head: int = N_HEAD
    n_embd: int = N_EMBD
    dropout: float = DROPOUT
    bias: bool = True # True: bias in Linears and LayerNorms

class LayerNorm(nn.Module):
    """ LayerNorm but with an optional bias. """
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # Key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # Output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # Regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd

        # Causal mask to ensure that attention is only applied to the left in the input sequence
        # We use register_buffer for parameters that should be part of the model's state
        # but are not trained by the optimizer (e.g., a fixed mask).
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # Calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # Manual implementation of attention
        # (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf')) # Apply causal mask
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C) # Re-assemble all head outputs side by side

        # Output projection
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU() # Using GELU activation
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))  # Attention with residual connection
        x = x + self.mlp(self.ln_2(x))   # MLP with residual connection
        return x

In [20]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),      # Token embeddings
            wpe = nn.Embedding(config.block_size, config.n_embd),     # Positional embeddings
            drop = nn.Dropout(config.dropout),                        # Dropout layer
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]), # Transformer blocks
            ln_f = LayerNorm(config.n_embd, bias=config.bias),        # Final layer norm
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False) # Language model head

        # Weight tying: token embeddings and final linear layer share weights
        self.transformer.wte.weight = self.lm_head.weight

        # Initialize weights
        self.apply(self._init_weights)
        # Apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        num_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Number of trainable parameters: {num_params/1e6:.2f}M")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size() # Batch size, sequence length
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device) # Shape (t)

        # Forward the GPT model
        tok_emb = self.transformer.wte(idx) # Token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # Position embeddings of shape (t, n_embd)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            # If we are given some desired targets also calculate the loss
            logits = self.lm_head(x) # (b, t, vocab_size)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            # Inference-time optimization: only forward the lm_head on the very last position
            logits = self.lm_head(x[:, [-1], :]) # Note: using list [-1] to preserve the time dim -> (b, 1, vocab_size)
            loss = None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        """
        self.eval() # Set model to evaluation mode
        for _ in range(max_new_tokens):
            # If the sequence context is growing too long, crop it at block_size
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            # Forward the model to get the logits for the index in the sequence
            logits, _ = self(idx_cond) # Loss is None during generation
            # Pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # Optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf') # Mask non-top-k logits
            # Apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # Append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)
        self.train() # Set model back to training mode if used elsewhere
        return idx

In [21]:
# --- Model Initialization and Initial Exploration ---

# Create GPTConfig instance
model_config = GPTConfig(vocab_size=vocab_size, block_size=BLOCK_SIZE,
                         n_layer=N_LAYER, n_head=N_HEAD, n_embd=N_EMBD, dropout=DROPOUT)

# Instantiate the model
model = GPT(model_config)
model.to(DEVICE)

# Get a batch of data
xb, yb = get_batch('train')

print("--- Before Training ---")
# Pass the batch through the UNTRAINED model
logits, loss = model(xb, yb)

print("Logits shape:", logits.shape) # Should be (BATCH_SIZE, BLOCK_SIZE, vocab_size)
print("Initial loss:", loss.item())

# For a randomly initialized model, the loss should be roughly -ln(1/vocab_size)
# This is because the model initially assigns roughly equal probability to each token in the vocabulary.
# The cross-entropy loss for a uniform distribution over V classes is -sum( (1/V) * log(1/V) ) = -V * (1/V) * log(1/V) = -log(1/V) = log(V)
expected_loss = -math.log(1.0 / vocab_size)
print(f"Expected initial loss (approx -ln(1/vocab_size)): {expected_loss:.4f}")
# The actual initial loss will vary due to specific weight initialization but should be in this ballpark.

# Let's look at the logits for the first token prediction in the first sequence
first_token_logits = logits[0, 0, :]
print("Logits for the first token prediction in the first sequence (first 10 values):", first_token_logits[:10])
# These are raw, unnormalized scores for each possible next character.
# After softmax, these would turn into probabilities.

Number of trainable parameters: 10.85M
--- Before Training ---
Logits shape: torch.Size([64, 256, 275])
Initial loss: 5.661283493041992
Expected initial loss (approx -ln(1/vocab_size)): 5.6168
Logits for the first token prediction in the first sequence (first 10 values): tensor([-0.2189,  0.5946, -0.1316, -0.3270,  0.1807,  0.0861, -0.0642, -0.0225,
         0.1260, -0.6683], device='cuda:0', grad_fn=<SliceBackward0>)


In [22]:
# --- Evaluation Function ---
@torch.no_grad() # Decorator to disable gradient calculations during evaluation
def estimate_loss(model_to_eval):
    out = {}
    model_to_eval.eval() # Set the model to evaluation mode
    for split in ['train', 'val']:
        losses = torch.zeros(EVAL_ITERS_FOR_LOSS) # Array to store losses for averaging
        for k in range(EVAL_ITERS_FOR_LOSS):
            X, Y = get_batch(split)
            _, current_loss = model_to_eval(X, Y)
            losses[k] = current_loss.item()
        out[split] = losses.mean() # Average loss over EVAL_ITERS_FOR_LOSS batches
    model_to_eval.train() # Set the model back to training mode
    return out

In [23]:

model_path = 'GPTmodel_trained_Newdataset.pth' # Or whatever you named it when uploading
if os.path.exists(model_path):
         model.load_state_dict(torch.load(model_path, map_location=DEVICE))
         model.eval() # Set to evaluation mode
         print(f"Model loaded from {model_path}")
else:
         print(f"Model file not found at {model_path}")

C:\Users\97596\AppData\Local\Temp\ipykernel_80632\3888893185.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=DE

Model loaded from GPTmodel_trained_Newdataset.pth


In [ ]:
# --- 4. Generate Text ---
print(f"\n--- Generating Fakenews text ---")

# You can change the starting prompt
start_string = "[input] "

print(f"Starting prompt: '{start_string.strip()}'")



start_ids = encode(start_string)
# Unsqueeze to add batch dimension: (seq_len) -> (1, seq_len)
x_input = torch.tensor(start_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

# Generate text
model.eval() # Set model to evaluation mode for generation
with torch.no_grad(): # No need to track gradients during generation
    generated_ids = model.generate(x_input,
                                   max_new_tokens=3000,
                                   temperature=0.7, # Controls randomness: lower is less random, higher is more random
                                   top_k=50)       # Considers only the top_k most likely tokens at each step

generated_text = decode(generated_ids[0].tolist()) # Decode the first (and only) batch item

#article = generated_text.split("=== END ===")[0] + "\n === END ==="


print("\n--- Generated Text ---")
print(generated_text)



--- Generating Fakenews text ---
Starting prompt: '[input]'

--- Generated Text ---
[input] Cairman city, Director John Kelly’s Data, May Reached - Breitbart && John Breitbart News Daily Collins, who started a recent deal among his meeting with companies and protesters who should be treated by the country s “biggest right” in April 100 People and Green s award and the report. But the third country s statement said in a statement were reported, and said. He is under values are concerned by federal election on Sunday allowed to the outcome of a big feel on Islamic State in 2015 which has been dropped in which they were perpetrative program, but it comes an attack on the case on deadly in a past the possibility of a bloc criminal advanced minority. Republican presidential nominee Hillary Clinton was also called for the deal to make a “committee to go to block along with the world s very day, which is the day, the accord to the president of senior President Trump and Huston SiriusXM Patri